# ID6004W Industrial Vision - Assignment 1
### Lectures 01-02: Image Filtering and Convolution

| | |
|---|---|
| **Released** | Oct 1st 2026 |
| **Due** | Oct 21st 2026 (5.00 PM IST) |
| **Total** | 100 marks (will be scaled down appropriately later) |

## Instructions

**Working in Colab.** Open this notebook in Google Colab and immediately use *File → Save a copy in Drive*; work only in your copy. Type your answers in the cells marked **Your answer**, directly below each question (double-click a cell to edit it). Mathematics can be typed in LaTeX between `$...$` or `$$...$$`. Photographs of hand-written working are **not** accepted in place of typed answers.

- **The writing must be your own.**
- Cells marked **Optional check** are not graded. Use them to verify a hand calculation numerically, but the answer you submit must be derived by hand with the steps shown.
-  **Submission.** When you are done, use *File → Download → Download .ipynb*, rename the file to `<ROLLNO>_assignment1.ipynb`, and upload it here: [SUBMISSION_LINK](https://forms.gle/pRwExrMB1dvKoU138). For queries contact Rishabh Shukla (da26e011@smail.iitm.ac.in).
- **Late policy:** 10 marks will be deducted for each day the assignment is submitted late after the due date.

## Student details

- **Name:** Amardeep Kumar
- **Roll number:** DA25M502

In [4]:
# Imports used by the optional check cells.
import numpy as np
from scipy import ndimage
rng = np.random.default_rng(0)

---
## Question 1. What is, and what is not, a convolution  [25 marks]

In Lecture 01 we proved that linearity together with shift invariance forces an operator to be a convolution. This question asks you to use that theorem as a *test*.

**(a)** For each operator $\mathcal{T}$ below, state whether it is (i) linear and (ii) shift invariant. If it is both, give its impulse response $h$ explicitly (as a small array, or as a formula). Here $f$ is the input image and $g=\mathcal{T}\{f\}$. **[15]**

1. $g(m,n) = f(m,n) - f(m-1,n)$
2. $g(m,n) = \operatorname{median}\big\{f(m+i,n+j) : |i|\le 1, |j|\le 1\big\}$
3. $g(m,n) = c(m,n)\,f(m,n)$, where $c$ is a fixed, non-constant mask (e.g. lens vignetting)
4. $g(m,n) = f(m,n)^{2}$
5. $g(m,n) = f(n,m)$ (transpose the image)
6. $g(m,n) = f(m,n) + 10$ (brighten)

### Answer 1(a).
An operator $\mathcal{T}$ is equivalent to a 2D convolution $g = f * h$ if and only if it satisfies both **linearity** and **shift invariance**.

For an operator $T$ mapping $f \to g$:

**Linearity:** $T\{a f_1 + b f_2\} = a T\{f_1\} + b T\{f_2\}$ for any scalars $a, b$.

**Shift Invariance:** If $f_{\Delta}(m,n) = f(m - m_0, n - n_0)$, then $T\{f_\Delta\}(m,n) = g(m - m_0, n - n_0)$.

**Impulse Response:** If both hold, $T$ is a 2D convolution $g = f * h$, where $h(m,n) = T\{\delta\}(m,n)$ and $\delta(m,n) = 1$ at $(0,0)$ and $0$ elsewhere.

---

#### 1. $g(m,n) = f(m,n) - f(m-1,n)$

* **(i) Linear:** **Yes**
  $$\mathcal{T}\{a f_1 + b f_2\}(m,n) = \big(a f_1(m,n) + b f_2(m,n)\big) - \big(a f_1(m-1,n) + b f_2(m-1,n)\big) = a \mathcal{T}\{f_1\}(m,n) + b \mathcal{T}\{f_2\}(m,n)$$
* **(ii) Shift Invariant:** **Yes**
  
  Let $f_{\Delta}(m,n) = f(m - m_0, n - n_0)$.
  $$\mathcal{T}\{f_{\Delta}\}(m,n) = f(m - m_0, n - n_0) - f(m - m_0 - 1, n - n_0) = g(m - m_0, n - n_0)$$
* **Impulse Response $h(m,n)$:**
  Applying $\mathcal{T}$ to the unit impulse $\delta(m,n)$:
  $$h(m,n) = \delta(m,n) - \delta(m-1,n)$$
  In matrix form (with origin $(0,0)$ at the top entry):
  $$h = \begin{bmatrix} 1 \\ -1 \end{bmatrix}$$

---

#### 2. $g(m,n) = \operatorname{median}\big\{f(m+i,n+j) : |i|\le 1, |j|\le 1\big\}$

* **(i) Linear:** **No**
  Median filtering violates both additivity and homogeneity:
  $$\operatorname{median}(f_1 + f_2) \neq \operatorname{median}(f_1) + \operatorname{median}(f_2)$$
  $$\operatorname{median}(a \cdot f) \neq a \cdot \operatorname{median}(f) \quad \text{for } a < 0$$
* **(ii) Shift Invariant:** **Yes**
  The neighborhood definition is uniform across the entire spatial grid:
  $$\mathcal{T}\{f(m-m_0, n-n_0)\} = \operatorname{median}\big\{f(m - m_0 + i, n - n_0 + j)\big\} = g(m-m_0, n-n_0)$$
* **Impulse Response $h(m,n)$:** **None** (operator is nonlinear).

---

#### 3. $g(m,n) = c(m,n)\,f(m,n)$, where $c$ is a fixed, non-constant mask

* **(i) Linear:** **Yes**
  $$\mathcal{T}\{a f_1 + b f_2\}(m,n) = c(m,n)\big(a f_1(m,n) + b f_2(m,n)\big) = a\,c(m,n)f_1(m,n) + b\,c(m,n)f_2(m,n) = a\mathcal{T}\{f_1\} + b\mathcal{T}\{f_2\}$$
* **(ii) Shift Invariant:** **No**
  For a shifted input $f_{\Delta}(m,n) = f(m - m_0, n - n_0)$:
  $$\mathcal{T}\{f_{\Delta}\}(m,n) = c(m,n)f(m - m_0, n - n_0) \neq c(m - m_0, n - n_0)f(m - m_0, n - n_0) = g(m - m_0, n - n_0)$$
  because $c(m,n)$ varies spatially with position.
* **Impulse Response $h(m,n)$:** **None** (operator is not shift invariant).

---

#### 4. $g(m,n) = f(m,n)^2$

* **(i) Linear:** **No**
  Squaring violates superposition:
  $$\mathcal{T}\{a f\}(m,n) = (a f(m,n))^2 = a^2 f(m,n)^2 \neq a\,\mathcal{T}\{f\}(m,n)$$
  $$\mathcal{T}\{f_1 + f_2\}(m,n) = (f_1 + f_2)^2 \neq f_1^2 + f_2^2$$
* **(ii) Shift Invariant:** **Yes**
  $$\mathcal{T}\{f(m-m_0, n-n_0)\} = \big(f(m-m_0, n-n_0)\big)^2 = g(m-m_0, n-n_0)$$
* **Impulse Response $h(m,n)$:** **None** (operator is nonlinear).

---

#### 5. $g(m,n) = f(n,m)$ (Transpose)

* **(i) Linear:** **Yes**
  $$\mathcal{T}\{a f_1 + b f_2\}(m,n) = a f_1(n,m) + b f_2(n,m) = a\,\mathcal{T}\{f_1\}(m,n) + b\,\mathcal{T}\{f_2\}(m,n)$$
* **(ii) Shift Invariant:** **No**
  Let $f_{\Delta}(m,n) = f(m - m_0, n - n_0)$.
  $$\mathcal{T}\{f_{\Delta}\}(m,n) = f_{\Delta}(n,m) = f(n - m_0, m - n_0)$$
  However, shifting the original output $g(m,n) = f(n,m)$ yields:
  $$g(m - m_0, n - n_0) = f(n - n_0, m - m_0)$$
  Since $f(n - m_0, m - n_0) \neq f(n - n_0, m - m_0)$ whenever $m_0 \neq n_0$, the operator is not shift invariant.
* **Impulse Response $h(m,n)$:** **None** (operator is not shift invariant).

---

#### 6. $g(m,n) = f(m,n) + 10$ (Brighten)

* **(i) Linear:** **No**
  It violates the zero-input condition ($\mathcal{T}\{0\} = 10 \neq 0$) and homogeneity:
  $$\mathcal{T}\{a f\}(m,n) = a f(m,n) + 10 \neq a\big(f(m,n) + 10\big)$$
* **(ii) Shift Invariant:** **Yes**
  $$\mathcal{T}\{f(m-m_0, n-n_0)\} = f(m-m_0, n-n_0) + 10 = g(m-m_0, n-n_0)$$
* **Impulse Response $h(m,n)$:** **None** (operator is nonlinear / affine).

In [5]:
# Optional check for 1(a). The six operators, with periodic boundaries so that shifts are exact.
N = 16
c_mask = rng.uniform(0.5, 1.0, (N, N))          # a fixed, non-constant mask

T = {
    1: lambda f: f - np.roll(f, 1, axis=0),
    2: lambda f: ndimage.median_filter(f, size=3, mode="wrap"),
    3: lambda f: c_mask * f,
    4: lambda f: f ** 2,
    5: lambda f: f.T,
    6: lambda f: f + 10,
}

def shift(f, dm, dn):
    return np.roll(f, (dm, dn), axis=(0, 1))

# TODO (optional): write is_linear(op) and is_shift_invariant(op) that test the two properties on
# random images f1, f2 and random scalars a, b, and print the result for every operator in T.

**(b)** Let $\mathcal{T}_1$ and $\mathcal{T}_2$ be two LSI systems with impulse responses $h_1$ and $h_2$. Show that the cascade $\mathcal{T}_2\{\mathcal{T}_1\{f\}\}$ is again LSI, identify its impulse response, and state why the order of the two systems does not matter. **[10]**

### Answer 1(b).

Let the cascade operator be defined as $\mathcal{T}\{f\} = \mathcal{T}_2\{\mathcal{T}_1\{f\}\}$.

---

#### (i) Linearity
Let $f = a f_A + b f_B$, where $a, b \in \mathbb{R}$.

1. By the linearity of $\mathcal{T}_1$:
   $$\mathcal{T}_1\{a f_A + b f_B\} = a \mathcal{T}_1\{f_A\} + b \mathcal{T}_1\{f_B\}$$

2. Applying $\mathcal{T}_2$ to this result, and using the linearity of $\mathcal{T}_2$:
   $$\mathcal{T}\{a f_A + b f_B\} = \mathcal{T}_2\big\{a \mathcal{T}_1\{f_A\} + b \mathcal{T}_1\{f_B\}\big\} = a \mathcal{T}_2\{\mathcal{T}_1\{f_A\}\} + b \mathcal{T}_2\{\mathcal{T}_1\{f_B\}\} = a \mathcal{T}\{f_A\} + b \mathcal{T}\{f_B\}$$

Hence, the cascade system $\mathcal{T}$ is **linear**.

---

#### (ii) Shift Invariance
Let $\mathcal{S}_{(m_0, n_0)}$ denote the spatial shift operator such that $\mathcal{S}_{(m_0, n_0)}\{f\}(m,n) = f(m - m_0, n - n_0)$.

1. Because $\mathcal{T}_1$ is shift-invariant, it commutes with the shift operator:
   $$\mathcal{T}_1\{\mathcal{S}_{(m_0, n_0)}\{f\}\} = \mathcal{S}_{(m_0, n_0)}\{\mathcal{T}_1\{f\}\}$$

2. Applying $\mathcal{T}_2$, and using the fact that $\mathcal{T}_2$ is also shift-invariant:
   $$\mathcal{T}\{\mathcal{S}_{(m_0, n_0)}\{f\}\} = \mathcal{T}_2\big\{\mathcal{T}_1\{\mathcal{S}_{(m_0, n_0)}\{f\}\}\big\} = \mathcal{T}_2\big\{\mathcal{S}_{(m_0, n_0)}\{\mathcal{T}_1\{f\}\}\big\} = \mathcal{S}_{(m_0, n_0)}\big\{\mathcal{T}_2\{\mathcal{T}_1\{f\}\}\big\} = \mathcal{S}_{(m_0, n_0)}\{\mathcal{T}\{f\}\}$$

Hence, the cascade system $\mathcal{T}$ is **shift-invariant**.

Since $\mathcal{T}$ is both linear and shift-invariant, it is an **LSI system**.

---

### 2. Identify the Impulse Response of the Cascade

By the fundamental theorem of LSI systems, any LSI system acts as a convolution with its impulse response:
$$\mathcal{T}_1\{f\} = f * h_1 \quad \text{and} \quad \mathcal{T}_2\{g\} = g * h_2$$

The cascade output for an arbitrary input $f$ is:
$$\mathcal{T}\{f\} = \mathcal{T}_2\{\mathcal{T}_1\{f\}\} = (f * h_1) * h_2$$

Using the associative property of convolution:
$$(f * h_1) * h_2 = f * (h_1 * h_2)$$

The impulse response $h$ of the cascade is the system's response to the unit impulse $\delta(m,n)$:
$$h = \mathcal{T}\{\delta\} = (\delta * h_1) * h_2 = h_1 * h_2$$

In explicit summation form:
$$h(m,n) = (h_1 * h_2)(m,n) = \sum_{k=-\infty}^{\infty} \sum_{l=-\infty}^{\infty} h_1(k,l) \, h_2(m-k, n-l)$$

---

### 3. Why the Order of the Two Systems Does Not Matter

The order of the systems does not matter because **convolution is commutative**:
$$h_1 * h_2 = h_2 * h_1$$

Consequently:
$$\mathcal{T}_2\{\mathcal{T}_1\{f\}\} = f * (h_1 * h_2) = f * (h_2 * h_1) = \mathcal{T}_1\{\mathcal{T}_2\{f\}\}$$

Thus, $\mathcal{T}_2 \circ \mathcal{T}_1 = \mathcal{T}_1 \circ \mathcal{T}_2$. Filtering with $h_1$ followed by $h_2$ produces the exact same result as filtering with $h_2$ followed by $h_1$.

---
## Question 2. Separability and rank  [25 marks]

**(a)** Determine the rank of each kernel below, and for those of rank one give an explicit factorisation $H=h_1h_2^{\top}$. Do this by hand, using the fact that a nonzero matrix has rank one exactly when every $2\times2$ minor vanishes. **[15]**

$$
A=\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
B=\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}
\quad
C=\begin{bmatrix}1&2&3\\2&4&6\\3&6&9\end{bmatrix}
\quad
D=\begin{bmatrix}1&1&1\\1&2&1\\1&1&1\end{bmatrix}
$$

### Answer 2(a).

### 1. Matrix $A$

$$A = \frac{1}{16}\begin{bmatrix}1 & 2 & 1\\ 2 & 4 & 2\\ 1 & 2 & 1\end{bmatrix}$$

* **Check $2 \times 2$ minors:**
  * Rows 1 & 2, Cols 1 & 2: $\det \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix} = 1(4) - 2(2) = 0$
  * Rows 1 & 2, Cols 2 & 3: $\det \begin{bmatrix} 2 & 1 \\ 4 & 2 \end{bmatrix} = 2(2) - 1(4) = 0$
  * Rows 2 & 3, Cols 1 & 2: $\det \begin{bmatrix} 2 & 4 \\ 1 & 2 \end{bmatrix} = 2(2) - 4(1) = 0$
  * Rows 2 & 3, Cols 2 & 3: $\det \begin{bmatrix} 4 & 2 \\ 2 & 1 \end{bmatrix} = 4(1) - 2(2) = 0$
  * Rows 1 & 3, Cols 1 & 3: $\det \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix} = 1(1) - 1(1) = 0$

Every $2 \times 2$ minor vanishes and $A \neq 0$.

* **Rank:** **$1$**
* **Factorisation ($A = h_1 h_2^\top$):**
  $$A = \left( \frac{1}{4}\begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \right) \left( \frac{1}{4}\begin{bmatrix} 1 & 2 & 1 \end{bmatrix} \right) = \frac{1}{16} \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \begin{bmatrix} 1 & 2 & 1 \end{bmatrix}$$

---

### 2. Matrix $B$

$$B = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$$

* **Check $2 \times 2$ minors:**
  * Take the top-left submatrix (Rows 1 & 2, Cols 1 & 2): $$\det \begin{bmatrix} 0 & 1 \\ 1 & -4 \end{bmatrix} = 0(-4) - 1(1) = -1 \neq 0$$

Since there exists a non-zero $2 \times 2$ minor, $\operatorname{rank}(B) \ge 2$.

* **Check $3 \times 3$ determinant:**
  $$\det(B) = -1 \cdot \det \begin{bmatrix} 1 & 1 \\ 0 & 0 \end{bmatrix} = 0$$

Because row 3 equals row 1 ($R_3 = R_1$), the rows are linearly dependent, so $\operatorname{rank}(B) < 3$.

* **Rank:** **$2$**
* **Factorisation:** Not rank one; no rank-1 factorisation exists.

---

### 3. Matrix $C$

$$C = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 3 & 6 & 9 \end{bmatrix}$$

* **Check $2 \times 2$ minors:**
  * Rows 1 & 2, Cols 1 & 2: $\det \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix} = 4 - 4 = 0$
  * Rows 1 & 2, Cols 2 & 3: $\det \begin{bmatrix} 2 & 3 \\ 4 & 6 \end{bmatrix} = 12 - 12 = 0$
  * Rows 2 & 3, Cols 1 & 2: $\det \begin{bmatrix} 2 & 4 \\ 3 & 6 \end{bmatrix} = 12 - 12 = 0$
  * Rows 2 & 3, Cols 2 & 3: $\det \begin{bmatrix} 4 & 6 \\ 6 & 9 \end{bmatrix} = 36 - 36 = 0$
  * Rows 1 & 3, Cols 1 & 3: $\det \begin{bmatrix} 1 & 3 \\ 3 & 9 \end{bmatrix} = 9 - 9 = 0$

Notice that $R_2 = 2 R_1$ and $R_3 = 3 R_1$. Every $2 \times 2$ minor vanishes and $C \neq 0$.

* **Rank:** **$1$**
* **Factorisation ($C = h_1 h_2^\top$):**
  $$C = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix} \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}$$

---

### 4. Matrix $D$

$$D = \begin{bmatrix} 1 & 1 & 1 \\ 1 & 2 & 1 \\ 1 & 1 & 1 \end{bmatrix}$$

* **Check $2 \times 2$ minors:**
  * Take the top-left submatrix (Rows 1 & 2, Cols 1 & 2): $$\det \begin{bmatrix} 1 & 1 \\ 1 & 2 \end{bmatrix} = 1(2) - 1(1) = 1 \neq 0$$

Since there is a non-zero $2 \times 2$ minor, $\operatorname{rank}(D) \ge 2$.

* **Check $3 \times 3$ determinant:**
  Row 1 and Row 3 are identical ($R_1 = R_3$), which makes $\det(D) = 0$, so $\operatorname{rank}(D) < 3$.

* **Rank:** **$2$**
* **Factorisation:** Not rank one; no rank-1 factorisation exists.

In [6]:
# Optional check for 2(a).
A = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16
B = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
C = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]])
D = np.array([[1, 1, 1], [1, 2, 1], [1, 1, 1]])

# TODO (optional): print the rank of each kernel, and for the rank-one kernels check that your
# factorisation np.outer(h1, h2) reproduces the kernel.

**(b)** An $N\times N$ image is smoothed with a Gaussian of $\sigma=5$, truncated to $2\lceil 3\sigma\rceil+1$ taps. State the kernel size $M$, the number of multiplications per pixel for a direct 2D pass and for two 1D passes, and the resulting speed-up. **[10]**

### Answer 2(b).

### 1. Kernel Size $M$

Given $\sigma = 5$, the formula for the number of taps along each axis is:

$$M = 2\lceil 3\sigma \rceil + 1 = 2\lceil 3 \times 5 \rceil + 1 = 2(15) + 1 = 31$$

The full 2D filter size is $31 \times 31$, so **$M = 31$**.

---

### 2. Multiplications Per Pixel

* **Direct 2D Pass:**
  Convolving an image directly with an $M \times M$ kernel requires computing an inner product of $M^2$ values for each output pixel:
  $$\text{Multiplications}_{\text{2D}} = M^2 = 31^2 = \mathbf{961} \text{ multiplications/pixel}$$

* **Two 1D Passes (Separable):**
  Because the Gaussian filter is separable ($G(x,y) = g(x) \cdot g(y)$), the 2D convolution can be split into two sequential 1D passes (a horizontal 1D pass of length $M$, followed by a vertical 1D pass of length $M$):
  $$\text{Multiplications}_{\text{1D+1D}} = M + M = 2M = 2 \times 31 = \mathbf{62} \text{ multiplications/pixel}$$

---

### 3. Resulting Speed-Up

The theoretical speed-up ratio achieved by exploiting separability is:

$$\text{Speed-up} = \frac{\text{Direct 2D Cost}}{\text{Two 1D Passes Cost}} = \frac{M^2}{2M} = \frac{M}{2} = \frac{31}{2} = \mathbf{15.5}$$

---
## Question 3. Noise gain  [25 marks]

Let the noise be i.i.d. with variance $\sigma_\eta^{2}$, so that a linear filter $w$ produces output noise variance $\sigma_\eta^{2}\sum_k w_k^{2}$.

**(a)** Compute $\sum_k w_k^{2}$ for each of the four kernels below, and give the output noise standard deviation as a multiple of $\sigma_\eta$. **[10]**

$$
\text{box } \tfrac19\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}
\quad
\text{Gaussian } \tfrac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
\text{Sobel } \begin{bmatrix}1&0&-1\\2&0&-2\\1&0&-1\end{bmatrix}
\quad
\text{sharpen } \begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}
$$

### Answer 3(a).

### Noise Variance Gain and Output Noise Standard Deviation

For an input image corrupted by i.i.d. noise with variance $\sigma_\eta^2$, a linear filter with weights $w$ produces an output noise variance of $\sigma_{\text{out}}^2 = \sigma_\eta^2 \sum_k w_k^2$, giving an output standard deviation of:

$$\sigma_{\text{out}} = \sqrt{\sum_k w_k^2} \cdot \sigma_\eta$$

---

#### 1. Box Filter
$$w = \frac{1}{9}\begin{bmatrix}1 & 1 & 1 \\ 1 & 1 & 1 \\ 1 & 1 & 1\end{bmatrix}$$

* **Sum of squared weights:**
  Each of the 9 entries equals $\frac{1}{9}$:
  $$\sum_k w_k^2 = 9 \times \left(\frac{1}{9}\right)^2 = 9 \times \frac{1}{81} = \frac{1}{9} \approx 0.1111$$

* **Output noise standard deviation:**
  $$\sigma_{\text{out}} = \sqrt{\frac{1}{9}}\,\sigma_\eta = \frac{1}{3}\sigma_\eta \approx \mathbf{0.3333\,\sigma_\eta}$$

---

#### 2. Gaussian Approximation
$$w = \frac{1}{16}\begin{bmatrix}1 & 2 & 1 \\ 2 & 4 & 2 \\ 1 & 2 & 1\end{bmatrix}$$

* **Sum of squared weights:**
  Summing the squared entries of the unscaled matrix:
  $$\sum (\text{unscaled})^2 = 4 \times (1^2) + 4 \times (2^2) + 1 \times (4^2) = 4(1) + 4(4) + 16 = 4 + 16 + 16 = 36$$
  Incorporating the scalar normalization factor $\frac{1}{16}$:
  $$\sum_k w_k^2 = \frac{36}{16^2} = \frac{36}{256} = \frac{9}{64} \approx 0.1406$$

* **Output noise standard deviation:**
  $$\sigma_{\text{out}} = \sqrt{\frac{9}{64}}\,\sigma_\eta = \frac{3}{8}\sigma_\eta = \mathbf{0.375\,\sigma_\eta}$$

---

#### 3. Sobel Filter
$$w = \begin{bmatrix}1 & 0 & -1 \\ 2 & 0 & -2 \\ 1 & 0 & -1\end{bmatrix}$$

* **Sum of squared weights:**
  $$\sum_k w_k^2 = 1^2 + 0^2 + (-1)^2 + 2^2 + 0^2 + (-2)^2 + 1^2 + 0^2 + (-1)^2$$
  $$\sum_k w_k^2 = 1 + 0 + 1 + 4 + 0 + 4 + 1 + 0 + 1 = \mathbf{12}$$

* **Output noise standard deviation:**
  $$\sigma_{\text{out}} = \sqrt{12}\,\sigma_\eta = 2\sqrt{3}\,\sigma_\eta \approx \mathbf{3.4641\,\sigma_\eta}$$

---

#### 4. Sharpen Filter
$$w = \begin{bmatrix}0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0\end{bmatrix}$$

* **Sum of squared weights:**
  $$\sum_k w_k^2 = 0^2 + (-1)^2 + 0^2 + (-1)^2 + 5^2 + (-1)^2 + 0^2 + (-1)^2 + 0^2$$
  $$\sum_k w_k^2 = 4 \times (-1)^2 + 5^2 = 4(1) + 25 = \mathbf{29}$$

* **Output noise standard deviation:**
  $$\sigma_{\text{out}} = \sqrt{29}\,\sigma_\eta \approx \mathbf{5.3852\,\sigma_\eta}$$

---

### Summary Table

| Kernel | $\sum_k w_k^2$ | Exact $\sigma_{\text{out}}$ | Numerical $\sigma_{\text{out}}$ | Effect on Noise |
| :--- | :---: | :---: | :---: | :--- |
| **Box** | $\frac{1}{9}$ | $\frac{1}{3}\sigma_\eta$ | $\approx 0.3333\,\sigma_\eta$ | Denoises (attenuates by $3$) |
| **Gaussian** | $\frac{9}{64}$ | $\frac{3}{8}\sigma_\eta$ | $0.375\,\sigma_\eta$ | Denoises (attenuates by $2.67$) |
| **Sobel** | $12$ | $2\sqrt{3}\sigma_\eta$ | $\approx 3.4641\,\sigma_\eta$ | Amplifies by $\approx 3.46$ |
| **Sharpen** | $29$ | $\sqrt{29}\sigma_\eta$ | $\approx 5.3852\,\sigma_\eta$ | Amplifies by $\approx 5.39$ |



In [7]:
# Optional check for 3(a): filter pure noise with each kernel and measure the output standard deviation.
kernels = {
    "box":      np.ones((3, 3)) / 9,
    "Gaussian": np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16,
    "Sobel":    np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]]),
    "sharpen":  np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}
sigma_eta = 1.0
noise = rng.normal(0, sigma_eta, (1024, 1024))

# TODO (optional): for each kernel compute sum(w**2), predict the output std, and compare it with the
# std of ndimage.convolve(noise, w, mode="wrap").

**(b)** Prove that among *all* $3\times3$ kernels whose weights sum to $1$, the box kernel minimises $\sum_k w_k^{2}$. (One line of Cauchy-Schwarz.) **[5]**

### Answer 3(b).

By the Cauchy–Schwarz inequality applied to the filter vector $\mathbf{w} = (w_1, \dots, w_9)^\top$ and the all-ones vector $\mathbf{1} = (1, \dots, 1)^\top$:

$$\left( \sum_{k=1}^9 w_k \cdot 1 \right)^2 \le \left( \sum_{k=1}^9 w_k^2 \right) \left( \sum_{k=1}^9 1^2 \right) \implies 1^2 \le \left( \sum_{k=1}^9 w_k^2 \right) \cdot 9 \implies \sum_{k=1}^9 w_k^2 \ge \frac{1}{9}$$

with equality holding if and only if $\mathbf{w}$ is proportional to $\mathbf{1}$ (i.e. $w_k = \frac{1}{9}$ for all $k$), which is precisely the box kernel.

**(c)** Your answers to (a) and (b) together say that the box filter is a strictly better denoiser, per pixel of support, than the $3\times3$ Gaussian. Yet the Gaussian is mostly the preferred choice. Give one distinct reason why. **[5]**

### Answer 3(c).

The Gaussian is preferred because it is **isotropic (rotationally symmetric)** and avoids **ringing/high-frequency artifacts** caused by the box filter's sharp edges.

* **In the spatial domain:** A box filter has sharp, square boundaries that treat diagonal neighbors differently from horizontal and vertical neighbors, causing blocky, directional artifacts. The Gaussian weights pixels smoothly and radially according to physical distance.
* **In the frequency domain:** The Fourier transform of a box filter is a 2D sinc function ($\operatorname{sinc}(u)\operatorname{sinc}(v)$), which has prominent side lobes with slow decay ($1/f$). These side lobes pass high-frequency noise and cause spurious oscillations (ringing artifacts). In contrast, the Fourier transform of a Gaussian is another Gaussian, which decays exponentially with zero side lobes and introduces no ringing.

**(d)** An alternative to spatial smoothing is to average $L$ frames of a static scene. How many frames are needed to match the noise reduction of one $3\times3$ box filter, and what does frame averaging give you that the box filter cannot? **[5]**

### Answer 3(d).

### 1. Number of Frames Needed

* **Noise Variance of a $3\times3$ Box Filter:**
  Applying a $3\times3$ box filter to i.i.d. noise with variance $\sigma_\eta^2$ yields an output variance of:
  $$\operatorname{Var}(\hat{f}_{\text{box}}) = \sigma_\eta^2 \sum_{k=1}^9 \left(\frac{1}{9}\right)^2 = \frac{\sigma_\eta^2}{9}$$

* **Noise Variance of Averaging $L$ Frames:**
  Averaging $L$ independent frames of a static scene reduces noise variance by a factor of $L$:
  $$\operatorname{Var}(\hat{f}_{\text{temp}}) = \frac{\sigma_\eta^2}{L}$$

Equating the two noise variances:
$$\frac{\sigma_\eta^2}{L} = \frac{\sigma_\eta^2}{9} \implies \mathbf{L = 9 \text{ frames}}$$

Exactly **$9$ frames** are required to match the noise reduction of a single $3\times3$ box filter.

---

### 2. What Frame Averaging Gives You That the Box Filter Cannot

Frame averaging preserves **full spatial resolution and high-frequency edge detail without any spatial blurring**:

* **Box Filter (Spatial Averaging):** Averages neighboring pixels within the same image, acting as a low-pass filter that blurs sharp edges, fine textures, and fine corners.
* **Frame Averaging (Temporal Averaging):** Averages the exact same spatial pixel across time. Since the underlying scene is static ($g_i(m,n) = f(m,n) + \eta_i(m,n)$), the expected value at every individual coordinate remains perfectly preserved:
  $$\mathbb{E}[\hat{f}(m,n)] = f(m,n)$$
  Consequently, it suppresses noise without sacrificing edge sharpness, spatial bandwidth, or introducing spatial distortion.

---
## Question 4. Zero-crossings  [25 marks]

**(a)** Show that the stencil $[1\;\;{-}2\;\;1]$ annihilates every affine ramp $f(x)=ax+b$, and that the five-point $\nabla^{2}$ annihilates every plane $f(x,y)=ax+by+c$. What does this tell you about the response of $\nabla^{2}$ inside a smoothly shaded region, and why is that desirable? **[15]**

### Answer 4(a).

### 1. The Stencil $[1 \;\; {-}2 \;\; 1]$ Annihilates Every Affine Ramp $f(x) = ax + b$

Let the input be an affine ramp $f(x) = ax + b$.

Applying the 1D discrete second-derivative stencil $[1 \;\; {-}2 \;\; 1]$ centered at $x$:

$$(f * [1 \;\; {-}2 \;\; 1])(x) = 1 \cdot f(x+1) - 2 \cdot f(x) + 1 \cdot f(x-1)$$

Substitute $f(x) = ax + b$:

$$\begin{aligned}
(f * [1 \;\; {-}2 \;\; 1])(x) &= \big(a(x+1) + b\big) - 2(ax + b) + \big(a(x-1) + b\big) \\
&= (ax + a + b) - (2ax + 2b) + (ax - a + b) \\
&= (ax + ax - 2ax) + (a - a) + (b + b - 2b) \\
&= 0 + 0 + 0 \\
&= \mathbf{0}
\end{aligned}$$

Thus, the stencil strictly evaluates to $0$ everywhere, annihilating any affine ramp regardless of the slope $a$ or intercept $b$.

---

### 2. The Five-Point $\nabla^2$ Annihilates Every Plane $f(x,y) = ax + by + c$

The standard discrete 5-point Laplacian kernel is:

$$\nabla^2 = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$$

Applying this stencil to an arbitrary planar function $f(x,y) = ax + by + c$:

$$(\nabla^2 f)(x,y) = f(x+1, y) + f(x-1, y) + f(x, y+1) + f(x, y-1) - 4 f(x,y)$$

Substitute $f(x,y)$:

$$\begin{aligned}
f(x+1, y) &= a(x+1) + by + c = ax + by + c + a \\
f(x-1, y) &= a(x-1) + by + c = ax + by + c - a \\
f(x, y+1) &= ax + b(y+1) + c = ax + by + c + b \\
f(x, y-1) &= ax + b(y-1) + c = ax + by + c - b \\
-4 f(x,y) &= -4(ax + by + c)
\end{aligned}$$

Adding all five terms together:

$$(\nabla^2 f)(x,y) = 4(ax + by + c) + (a - a + b - b) - 4(ax + by + c) = \mathbf{0}$$

Hence, the 5-point discrete Laplacian annihilates every plane $f(x,y) = ax + by + c$.

---

### 3. Response Inside a Smoothly Shaded Region & Why It Is Desirable

* **Response inside a smoothly shaded region:**  
  By Taylor's theorem, any smooth spatial illumination or shading gradient can be locally approximated to first order as an affine tangent plane:
  $$f(x_0 + \Delta x, y_0 + \Delta y) \approx f(x_0, y_0) + \frac{\partial f}{\partial x}\Delta x + \frac{\partial f}{\partial y}\Delta y$$
  Because the Laplacian is a second-order differential operator ($\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$) and annihilates planes, its response inside any smoothly shaded or linearly varying illumination region is **identically zero (or negligibly small)**.

* **Why this is desirable:**
  1. **Invariance to smooth lighting and shadows:** In edge detection and feature extraction, gradual brightness changes caused by diffuse shading, surface orientation, vignettes, or soft shadows are nuisance signals. Annihilating linear ramps ensures that edge detectors do not trigger false edges across uniformly or smoothly shaded surfaces.
  2. **Selectivity for rapid transitions:** Second derivatives respond only to **curvature** (changes in the gradient). Consequently, the Laplacian isolates true physical discontinuities (sharp boundaries, step edges, and fine structures) where higher-order derivatives are non-zero, producing sharp zero-crossings at boundaries while keeping homogenous and gently shaded backgrounds completely dark (zero response).

In [8]:
# Optional check for 4(a).
stencil_1d = np.array([1, -2, 1])
laplacian_5pt = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
x = np.arange(20.0)
X, Y = np.meshgrid(np.arange(20.0), np.arange(20.0))

# TODO (optional): apply the stencil to a ramp a*x + b and the five-point Laplacian to a plane
# a*X + b*Y + c (use mode="valid" / interior pixels only) and check that the result is zero.

**(b)** A 1D edge is sampled and the second difference takes the values

$$f''(62)=+1.07,\qquad f''(63)=+0.33,\qquad f''(64)=-0.49 .$$

Locate the zero-crossing to sub-pixel accuracy by linear interpolation, showing the arithmetic. The corresponding $|f'|$ values are $14.10,\,14.80,\,14.72$; state the best estimate of the edge location obtainable from $\arg\max|f'|$ alone. **[10]**

### Answer 4(b).

### 1. Zero-Crossing Location via Linear Interpolation

A zero-crossing of the second derivative $f''$ occurs between two consecutive samples where the sign changes.

Looking at the sampled values:
* $f''(62) = +1.07 > 0$
* $f''(63) = +0.33 > 0$
* $f''(64) = -0.49 < 0$

The sign change from positive to negative occurs strictly in the interval $[63, 64]$:
* At $x_1 = 63$: $y_1 = f''(63) = 0.33$
* At $x_2 = 64$: $y_2 = f''(64) = -0.49$

#### Linear Interpolation Arithmetic

Using linear interpolation between $(x_1, y_1)$ and $(x_2, y_2)$:

$$y(x) = y_1 + \frac{y_2 - y_1}{x_2 - x_1}(x - x_1)$$

Setting $y(x^*) = 0$ to find the zero-crossing $x^*$:

$$0 = y_1 + \frac{y_2 - y_1}{1}(x^* - x_1)$$

$$x^* - x_1 = \frac{-y_1}{y_2 - y_1} = \frac{y_1}{y_1 - y_2}$$

$$x^* = x_1 + \frac{y_1}{y_1 - y_2}$$

Substitute the numerical values:

$$x^* = 63 + \frac{0.33}{0.33 - (-0.49)}$$

$$x^* = 63 + \frac{0.33}{0.33 + 0.49} = 63 + \frac{0.33}{0.82}$$

$$\frac{0.33}{0.82} = \frac{33}{82} \approx 0.4024$$

$$x^* \approx 63 + 0.4024 = \mathbf{63.40} \quad (\text{or } 63.4024)$$

The estimated sub-pixel zero-crossing location is **$x^* \approx 63.40$**.

---

### 2. Best Estimate from $\arg\max|f'|$ Alone

The sampled gradient magnitudes $|f'|$ are:
* $|f'(62)| = 14.10$
* $|f'(63)| = 14.80$
* $|f'(64)| = 14.72$

Finding the integer coordinate that maximizes $|f'|$:

$$\arg\max_{x \in \{62, 63, 64\}} |f'(x)| = \mathbf{63} \quad (\text{since } 14.80 > 14.72 > 14.10)$$

The best estimate of the edge location obtainable from $\arg\max|f'|$ alone (at integer pixel resolution) is **$x = 63$**.

In [9]:
# Optional check for 4(b).
xs = np.array([62, 63, 64])
f2 = np.array([1.07, 0.33, -0.49])     # second difference
f1 = np.array([14.10, 14.80, 14.72])   # |first difference|

# TODO (optional): compute the linearly interpolated zero-crossing of f2 and xs[argmax(f1)].

---
*End of Assignment 1.*